# Leistungsverzeichnis (LV) Extraction & Structured Data Table

This notebook extracts and parses German Construction Bill of Quantities (*Leistungsverzeichnis* / LV) from PDF files into a structured tabular `pandas.DataFrame`.

The final table includes:
- **`oz`**: Ordnungszahl (position number)
- **`position_type`**: Position classification (`Normalposition`, `Wahlposition`, `Grundposition`, `Bedarfsposition`)
- **`short_text`**: Short title / summary of the work item
- **`quantity`**: Numerical quantity
- **`unit`**: Unit of measurement (e.g. `m²`, `m`, `St`, `h`)
- **`only_unit_price`**: Boolean indicating whether only a unit price applies (`Nur Einh.-Pr.`)
- **`long_text`**: Detailed description / specifications


In [1]:
import os
import re
import pdfplumber
import pandas as pd

# Set display options for clear table visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 1000)


In [2]:
# Regex patterns tailored to German LV format and resilient to spacing/kerning
OZ_PATTERN = re.compile(
    r"(?m)^\s*(\.\s*\d+(?:\s*\d+)*|\d+\.\d+\.(?:\s*\d+\.?)*)\s+"
)
QTY_UNIT_PATTERN = re.compile(
    r"(\d+(?:\s*\d+)*(?:\.\d{3})*,\d{3})\s*(m²|m|St|Set|Psch|psch|Satz|h)(?!\w)"
)
NUR_EINH_PR_PATTERN = re.compile(r"N\s*ur\s*Einh\s*\.\s*-\s*Pr\s*\.", re.IGNORECASE)

NOISE_PATTERNS = [
    r"(?i)Übertrag:.*",
    r"(?i)Ü\s*bertrag:.*",
    r"(?i)Druckdatum:.*",
    r"(?i)Seite:\s*\d+.*",
    r"(?i)S\s*umme\s+LV.*",
    r"(?i)OZ\s+L\s*eistungsbeschreibung.*",
    r"(?i)i\s*n\s+EUR\s+in\s+EUR.*",
    r"(?i)Z\s*uzüglich der gesetzlichen Mehrwertsteuer.*",
    r"(?i)i\s*n\s+Höhe von.*",
    r"(?i)\(O\s*rt\)\s*\(.*Datum\).*",
    r"(?i)Das LV besteht aus den Seiten.*",
    r"(?m)^\s*[\.\s]{3,}\s*EUR.*",
]


def extract_clean_text(pdf_path: str, top_margin: int = 145, bottom_margin: int = 810) -> str:
    """Extracts text while cropping out repeating headers and footers."""
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # Crop box: (x0, top, x1, bottom) - top_margin=145 removes header, bottom_margin=810 keeps quantities
            bbox = (0, top_margin, page.width, min(bottom_margin, page.height))
            cropped = page.crop(bbox)
            text = cropped.extract_text(layout=False) or ""
            pages_text.append(text)

    full_text = "\n".join(pages_text)
    for pattern in NOISE_PATTERNS:
        full_text = re.sub(pattern, "", full_text)
    return full_text


def parse_german_float(num_str: str) -> float:
    """Converts '3.300,000' or '9 4,000' -> 3300.0 / 94.0"""
    cleaned = re.sub(r"\s+", "", num_str)
    return float(cleaned.replace(".", "").replace(",", "."))


def parse_lv_positions(full_text: str) -> pd.DataFrame:
    """Splits continuous LV text into structured rows per OZ."""
    matches = list(OZ_PATTERN.finditer(full_text))
    records = []

    for i, match in enumerate(matches):
        oz_raw = match.group(1)
        oz = re.sub(r"\s+", "", oz_raw)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        block = full_text[start:end].strip()

        # Detect position type
        pos_type = "Normalposition"
        if "Wahlposition" in block:
            pos_type = "Wahlposition"
        elif "Bedarfsposition" in block:
            pos_type = "Bedarfsposition"
        elif "Grundposition" in block:
            pos_type = "Grundposition"

        only_unit_price = bool(NUR_EINH_PR_PATTERN.search(block))

        qty_match = QTY_UNIT_PATTERN.search(block)
        quantity = parse_german_float(qty_match.group(1)) if qty_match else None
        unit = qty_match.group(2) if qty_match else None

        lines = [line.strip() for line in block.splitlines() if line.strip()]
        title_idx = 1 if lines and ("position" in lines[0].lower()) else 0
        short_text = lines[title_idx] if len(lines) > title_idx else ""
        long_text = "\n".join(lines[title_idx + 1 :]).strip()

        records.append({
            "oz": oz,
            "position_type": pos_type,
            "short_text": short_text,
            "quantity": quantity,
            "unit": unit,
            "only_unit_price": only_unit_price,
            "long_text": long_text,
        })

    return pd.DataFrame(records)


## 1. Extract and Parse the PDF

In [3]:
# Locate target PDF
pdf_filename = "LV_14a_Betonkernaktivierung.pdf"
candidates = [
    pdf_filename,
    os.path.join("..", "Data", pdf_filename),
    os.path.join("Data", pdf_filename),
]
pdf_path = next((c for c in candidates if os.path.exists(c)), None)

if not pdf_path:
    raise FileNotFoundError(f"Could not find {pdf_filename}")

print(f"Loading PDF from: {pdf_path}")
raw_text = extract_clean_text(pdf_path)
print(f"Total raw characters extracted: {len(raw_text):,}")

df = parse_lv_positions(raw_text)
print(f"Parsed {len(df)} positions successfully.")


Loading PDF from: Data\LV_14a_Betonkernaktivierung.pdf


Total raw characters extracted: 23,939
Parsed 36 positions successfully.


## 2. The Final Parsed Table

This is the main structured dataset produced by the parser:


In [4]:
# Display the complete table without long_text clutter
summary_columns = ["oz", "position_type", "short_text", "quantity", "unit", "only_unit_price"]
df[summary_columns]


,oz,position_type,short_text,quantity,unit,only_unit_price
0,.1,Grundposition,Rohrregister zur Betonkernaktivierung in Modulbauweise,496.0,m²,False
1,.2,Wahlposition,Alternativ : Handverlegtes PE- Xa - Rohr mit allem Zubehör,3300.0,m,True
2,.3,Normalposition,Anbindeleitung ( für die Bauteilaktivierung),510.0,m,False
3,.4,Normalposition,Anbindeleitung ( für die Thermischen Steckdosen ),680.0,m,False
4,.5,Normalposition,Schutzrohr für Anbindeleitung,94.0,m,False
5,.6,Normalposition,"Thermische Steckdose für PE- Xa Rohr 2 0 x2 ,3",47.0,St,False
6,.7,Normalposition,Adapterstecker für Thermische Steckdose,47.0,St,False
7,.8,Normalposition,Uponor Press- Kupplung PE - Xa 2 0-20 mit 2 Presshülsen,94.0,St,False
8,.9,Normalposition,"Heiz- u nd Kühlkreisverteiler, 8 Kreise",1.0,St,False
9,.10,Normalposition,"Heiz- u nd Kühlkreisverteiler, 7 Kreise",3.0,St,False


## 3. Position Types & Quantities Breakdown

In [5]:
# Summary statistics
print("=== Position Types Count ===")
print(df["position_type"].value_counts())

print("\n=== Units Count ===")
print(df["unit"].value_counts(dropna=False))

print("\n=== Total Quantities by Unit ===")
print(df.groupby("unit")["quantity"].agg(["count", "sum", "min", "max"]))


=== Position Types Count ===
position_type
Normalposition     28
Bedarfsposition     6
Grundposition       1
Wahlposition        1
Name: count, dtype: int64

=== Units Count ===
unit
St      21
m        5
h        3
Set      2
Psch     2
m²       1
psch     1
Satz     1
Name: count, dtype: int64

=== Total Quantities by Unit ===
      count     sum    min     max
unit                              
Psch      2     2.0    1.0     1.0
Satz      1     1.0    1.0     1.0
Set       2    16.0    8.0     8.0
St       21   862.0    1.0    94.0
h         3     3.0    1.0     1.0
m         5  4678.0   94.0  3300.0
m²        1   496.0  496.0   496.0
psch      1     1.0    1.0     1.0


## 4. Inspecting Full Position Details (Long Text)

In [6]:
# Preview first item with long specification description
first_row = df.iloc[0]
print(f"OZ:            {first_row['oz']}")
print(f"Type:          {first_row['position_type']}")
print(f"Short text:    {first_row['short_text']}")
print(f"Quantity:      {first_row['quantity']} {first_row['unit']}")
print(f"Unit price:    {first_row['only_unit_price']}")
print(f"\n--- Long Text Description ---\n{first_row['long_text'][:400]}...")


OZ:            .1
Type:          Grundposition
Short text:    Rohrregister zur Betonkernaktivierung in Modulbauweise
Quantity:      496.0 m²
Unit price:    False

--- Long Text Description ---
( Einbausituation Betondecke)
Rohrregister in Form von vorgefertigten Modulen mit
integrierten Anbindeleitungen individueller Länge zur
Montage zwischen bauseitiger unterer Bewehrung und
oberer Bewehrung zur thermischen Aktivierung der
Betondecke, b estehend aus :
- w erkseitig objektbezogen komplett vorgefertigtem
Modul ,
- P E- X a Rohr 2 0 x 2 ,3 mm aus hochdruckvernetztem
Polyethylen , mit 5 -...


## 5. Export Structured Data

In [7]:
# Export to CSV and JSON
csv_path = "lv_structured.csv"
json_path = "lv_structured.json"

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_json(json_path, orient="records", force_ascii=False, indent=2)

print(f"Exported {len(df)} rows:")
print(f"  - CSV:  {csv_path} ({os.path.getsize(csv_path):,} bytes)")
print(f"  - JSON: {json_path} ({os.path.getsize(json_path):,} bytes)")


Exported 36 rows:
  - CSV:  lv_structured.csv (21,478 bytes)
  - JSON: lv_structured.json (26,615 bytes)
